In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, MapType
from datetime import date

# Assume spark is already created
spark = SparkSession.builder.appName("PracticeDataset").getOrCreate()

# Define the schema
schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", DateType(), True),
    StructField("tags", StringType(), True), # For split() and regex examples
    StructField("details", StringType(), True), # For translate(), substring() examples
    StructField("features", StringType(), True), # For regexp_extract(), regexp_replace() examples
    StructField("product_attributes", MapType(StringType(), StringType()), True), # For MapType, explode, map_keys, map_values
    StructField("related_products", StringType(), True) # For collect_list, collect_set
])

# Sample data
data = [
    (1, 101, "Laptop", 1, 1200.00, date(2023, 1, 15), "electronics,office", "Serial: A1B2C3D4", "user_123_laptop", {"color": "silver", "brand": "XYZ"}, "Keyboard,Mouse,Monitor"),
    (2, 102, "Mouse", 2, 25.50, date(2023, 1, 15), "electronics,accessory", "Model: M500", "user_456_mouse", {"color": "black", "wireless": "true"}, "Laptop,Monitor"),
    (3, 101, "Keyboard", 1, 75.00, date(2023, 1, 16), "electronics,accessory", "SKU: KB-101", "id_789_keyboard", {"layout": "US", "mechanical": "false"}, "Mouse,Monitor"),
    (4, 103, "Monitor", 1, 300.00, date(2023, 1, 16), "electronics,display", "DisplaySize: 27inch", "user_123_monitor", {"size": "27", "resolution": "1080p"}, "Laptop,Keyboard"),
    (5, 102, "Desk Chair", 1, 150.00, date(2023, 1, 17), "furniture,office", "Weight: 20kg", "user_456_chair", {"material": "mesh", "adjustable": "true"}, "Desk,Lamp"),
    (6, 104, "Lamp", 2, 35.00, date(2023, 1, 17), "furniture,lighting", "BulbType: LED", "id_abc_lamp", None, "Desk Chair,Desk"), # Example with None map
    (7, 101, "Laptop", 1, 1200.00, date(2023, 1, 18), "electronics,office", "Serial: E5F6G7H8", "user_123_laptop", {"color": "silver", "brand": "XYZ"}, "Keyboard,Mouse,Monitor"), # Duplicate order
    (8, 105, "Notebook", 5, 3.00, date(2023, 1, 18), "office,stationery", "Pages: 100", "user_xyz_notebook", {}, "Pen,Pencil"), # Example with empty map
    (9, 103, "Desk", 1, 250.00, date(2023, 1, 19), "furniture,office", "Material: Wood", "user_789_desk", {"size": "medium"}, "Desk Chair,Lamp"),
    (10, 104, "Pen", 10, 1.50, date(2023, 1, 19), "office,stationery", "InkColor: Blue", "id_def_pen", {"color": "blue"}, "Notebook,Pencil"),
    (11, 105, "Pencil", 12, 0.50, date(2023, 1, 20), "office,stationery", "LeadSize: 0.7mm", "user_xyz_pencil", {"lead": "0.7"}, "Notebook,Pen"),
    (12, 106, "Tablet", 1, 400.00, date(2023, 1, 20), "electronics", "Model: Tab-Pro", "user_abc_tablet", {"os": "Android"}, None), # Example with None related_products
    (13, 106, "Protector", 1, 15.00, date(2023, 1, 20), "electronics,accessory", "Type: Screen", "user_abc_protector", {}, ""), # Example with empty related_products
    # Added new rows
    (14, 101, "Mouse", 1, 25.50, date(2023, 1, 21), "electronics,accessory", "Model: M600", "user_101_mouse", {"color": "white", "wireless": "false"}, "Keyboard"),
    (15, 102, "Laptop", 1, 1100.00, date(2023, 1, 21), "electronics,office", "Serial: I9J10K11L12", "user_102_laptop", {"color": "black", "brand": "UVW"}, "Mouse,Monitor"),
    (16, 103, "Keyboard", 2, 70.00, date(2023, 1, 22), "electronics,accessory", "SKU: KB-202", "user_103_keyboard", {"layout": "UK", "mechanical": "true"}, "Mouse"),
    (17, 104, "Desk", 1, 220.00, date(2023, 1, 22), "furniture,office", "Material: Metal", "user_104_desk", {"size": "large"}, "Chair"),
    (18, 105, "Lamp", 1, 30.00, date(2023, 1, 23), "furniture,lighting", "BulbType: Incandescent", "user_105_lamp", None, "Desk"),
    (19, 106, "Notebook", 3, 2.50, date(2023, 1, 23), "office,stationery", "Pages: 150", "user_106_notebook", {}, "Pen,Pencil"),
    (20, 101, "Monitor", 1, 280.00, date(2023, 1, 24), "electronics,display", "DisplaySize: 24inch", "user_101_monitor", {"size": "24", "resolution": "1080p"}, "Laptop")
]

practice_df = spark.createDataFrame(data, schema)

# Show the DataFrame and its schema
print("Sample Practice DataFrame:")
practice_df.show(truncate=False)
practice_df.printSchema()

Sample Practice DataFrame:
+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+
|order_id|customer_id|product   |quantity|price |order_date|tags                 |details               |features          |product_attributes                    |related_products      |
+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+
|1       |101        |Laptop    |1       |1200.0|2023-01-15|electronics,office   |Serial: A1B2C3D4      |user_123_laptop   |{color -> silver, brand -> XYZ}       |Keyboard,Mouse,Monitor|
|2       |102        |Mouse     |2       |25.5  |2023-01-15|electronics,accessory|Model: M500           |user_456_mouse    |{color -> black, wireless -> true}    |Laptop,Monitor        |
|3       |101        |Keyboard  |1    

In [ ]:
from os import truncate
from pyspark.sql.functions import col,explode,split
df_exploded_tags=practice_df.select(col("order_id"),explode(split(col("tags"),",")).alias("tag"))
df_exploded_tags.show()

+--------+-----------+
|order_id|        tag|
+--------+-----------+
|       1|electronics|
|       1|     office|
|       2|electronics|
|       2|  accessory|
|       3|electronics|
|       3|  accessory|
|       4|electronics|
|       4|    display|
|       5|  furniture|
|       5|     office|
|       6|  furniture|
|       6|   lighting|
|       7|electronics|
|       7|     office|
|       8|     office|
|       8| stationery|
|       9|  furniture|
|       9|     office|
|      10|     office|
|      10| stationery|
+--------+-----------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import regexp_extract
df_details_serial=practice_df.withColumn("serial_number",regexp_extract(col("details"),r"Serial:(.*)",1))
df_details_serial.show()

+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+-------------+
|order_id|customer_id|   product|quantity| price|order_date|                tags|             details|          features|  product_attributes|    related_products|serial_number|
+--------+-----------+----------+--------+------+----------+--------------------+--------------------+------------------+--------------------+--------------------+-------------+
|       1|        101|    Laptop|       1|1200.0|2023-01-15|  electronics,office|    Serial: A1B2C3D4|   user_123_laptop|{color -> silver,...|Keyboard,Mouse,Mo...|     A1B2C3D4|
|       2|        102|     Mouse|       2|  25.5|2023-01-15|electronics,acces...|         Model: M500|    user_456_mouse|{color -> black, ...|      Laptop,Monitor|             |
|       3|        101|  Keyboard|       1|  75.0|2023-01-16|electronics,acces...|         SKU: KB-101|   id_78

In [ ]:
from os import truncate
from pyspark.sql.functions import col,explode,split,trim
d
df=practice_df.withColumn('details',df_exploded_tags1.getItem(1))
df_exploded_tags1.show()
df.show()

NameError: name 'd' is not defined

In [ ]:
from pyspark.sql.functions import explode,map_keys,map_values,explode_outer
df_exploded_attributes=practice_df.select(col("order_id"),col("product"),explode_outer(col("product_attributes")).alias("att_key","att_values"))
df_exploded_attributes.show()


+--------+----------+----------+----------+
|order_id|   product|   att_key|att_values|
+--------+----------+----------+----------+
|       1|    Laptop|     color|    silver|
|       1|    Laptop|     brand|       XYZ|
|       2|     Mouse|     color|     black|
|       2|     Mouse|  wireless|      true|
|       3|  Keyboard|    layout|        US|
|       3|  Keyboard|mechanical|     false|
|       4|   Monitor|      size|        27|
|       4|   Monitor|resolution|     1080p|
|       5|Desk Chair|  material|      mesh|
|       5|Desk Chair|adjustable|      true|
|       6|      Lamp|      NULL|      NULL|
|       7|    Laptop|     color|    silver|
|       7|    Laptop|     brand|       XYZ|
|       8|  Notebook|      NULL|      NULL|
|       9|      Desk|      size|    medium|
|      10|       Pen|     color|      blue|
|      11|    Pencil|      lead|       0.7|
|      12|    Tablet|        os|   Android|
|      13| Protector|      NULL|      NULL|
|      14|     Mouse|     color|

In [ ]:

from os import truncate
from pyspark.sql.functions import col,explode,split,trim
practice_df = practice_df.withColumn("details_key", trim(split(col("details"), ":")[0])) \
                         .withColumn("details_value", trim(split(col("details"), ":")[1]))

practice_df.show(truncate=False)

+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+-----------+-------------+
|order_id|customer_id|product   |quantity|price |order_date|tags                 |details               |features          |product_attributes                    |related_products      |details_key|details_value|
+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+-----------+-------------+
|1       |101        |Laptop    |1       |1200.0|2023-01-15|electronics,office   |Serial: A1B2C3D4      |user_123_laptop   |{color -> silver, brand -> XYZ}       |Keyboard,Mouse,Monitor|Serial     |A1B2C3D4     |
|2       |102        |Mouse     |2       |25.5  |2023-01-15|electronics,accessory|Model: M500           |user_456_mouse    |{color -> black, wireles

In [ ]:

from os import truncate
from pyspark.sql.functions import col,explode,split,trim,expr
practice_df = practice_df.withColumn("colon_pos", expr("instr(details, ':')"))

# Extract left part
practice_df = practice_df.withColumn("details_key", expr("trim(left(details, colon_pos - 1))"))

# Extract right part
practice_df = practice_df.withColumn("details_value", expr("trim(substr(details, colon_pos + 1, length(details)))"))

practice_df.show(truncate=False)

+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+-----------+-------------+---------+
|order_id|customer_id|product   |quantity|price |order_date|tags                 |details               |features          |product_attributes                    |related_products      |details_key|details_value|colon_pos|
+--------+-----------+----------+--------+------+----------+---------------------+----------------------+------------------+--------------------------------------+----------------------+-----------+-------------+---------+
|1       |101        |Laptop    |1       |1200.0|2023-01-15|electronics,office   |Serial: A1B2C3D4      |user_123_laptop   |{color -> silver, brand -> XYZ}       |Keyboard,Mouse,Monitor|Serial     |A1B2C3D4     |7        |
|2       |102        |Mouse     |2       |25.5  |2023-01-15|electronics,accessory|Model: M500           |use

In [ ]:
InventoryControl_ETL/
│
├── data/
│   ├── sales/                # Streaming sales data (JSON files)
│   │    ├── sales_1.json
│   │    ├── sales_2.json
│   │    └── ...
│   └── stock.csv             # Current inventory stock
│
├── scripts/
│   └── etl_inventory.py      # PySpark ETL code
│
├── requirements.txt          # Python dependencies
└── README.md


SyntaxError: invalid character '│' (U+2502) (ipython-input-1355348866.py, line 2)

In [ ]:
from kafka import KafkaProducer
import json
import time
import random
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

products = ["B001", "B002", "B003", "B004", "B005"]
stores = ["S001", "S002", "S003"]

while True:
    sale = {
        "transaction_id": f"T{random.randint(1000,9999)}",
        "store_id": random.choice(stores),
        "product_id": random.choice(products),
        "quantity_sold": random.randint(1,5),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    producer.send('bata_sales', sale)
    print(f"Sent: {sale}")
    time.sleep(2)


ModuleNotFoundError: No module named 'kafka'